In [429]:
from datetime import datetime
import re
from contextlib import ExitStack
from enum import Enum
from typing import Optional

from deepagents import SubAgent, create_deep_agent, FilesystemPermission
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend, FilesystemBackend
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.tools import tool
from dotenv import load_dotenv
from pathlib import Path
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore

from langgraph.prebuilt import ToolRuntime
from langchain_postgres import PGVectorStore, Column
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langgraph.stream import GraphRunStream
from pydantic import BaseModel



In [430]:
load_dotenv()

True

In [431]:
POSTGRES_USER = "postgres"
POSTGRES_PASSWORD = "7db8cb60"
POSTGRES_HOST = "localhost"
POSTGRES_PORT = 5432
POSTGRES_DB = "carlosdaniel"
TABLE_NAME = "man_united_source_of_truth"
CONNECTION_STRING = (
    f"postgresql+psycopg://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}"
    f":{POSTGRES_PORT}/{POSTGRES_DB}"
)
from langchain_postgres import PGEngine

pg_engine = PGEngine.from_connection_string(url=CONNECTION_STRING)

In [329]:
pg_engine.init_vectorstore_table(
    table_name=TABLE_NAME,
    vector_size=3072,
    metadata_columns=[
        Column("source", "TEXT"),
        Column("type", "VARCHAR(20)"),
        Column("path", "TEXT")
    ]
)

In [432]:
model = ChatOpenAI(model="gpt-6.1-sol")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [433]:
store = PGVectorStore.create_sync(
    engine=pg_engine,
    table_name=TABLE_NAME,
    embedding_service=embeddings,
    metadata_columns=["source", "type", "path"]
)

In [434]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, chunk_overlap=300, add_start_index=True
)
markdown_splitter = MarkdownHeaderTextSplitter(
    [("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")]
)

In [435]:
class Context(BaseModel):
    user_id: str

In [436]:
class SourceTypes(Enum):
    HOMEPAGE = 'homepage'
    NEWS = 'news'
    VIDEOS = 'videos'
    STORE = 'store'
    MATCHES = 'matches'
    PEOPLE = 'people'
    MEMBERSHIP = 'membership'
    TRAVEL = 'travel'

In [437]:
class SourcePydantic(BaseModel):
    type: SourceTypes

In [438]:
DIRECTORIES = {
    'homepage': SourceTypes.HOMEPAGE,
    'matches': SourceTypes.MATCHES,
    'matches/reports': SourceTypes.MATCHES,
    'membership': SourceTypes.MEMBERSHIP,
    'news': SourceTypes.NEWS,
    'news/articles': SourceTypes.NEWS,
    'news/pages': SourceTypes.NEWS,
    'people': SourceTypes.PEOPLE,
    'people/men': SourceTypes.PEOPLE,
    'people/women': SourceTypes.PEOPLE,
    'people/staff': SourceTypes.PEOPLE,
    'store/catalog': SourceTypes.STORE,
    'store/products': SourceTypes.STORE,
    'travel': SourceTypes.TRAVEL,
    'videos': SourceTypes.VIDEOS,
    'videos/women': SourceTypes.VIDEOS
}

def load_pagecontent_with_headers(md_split_docs):
    if md_split_docs.metadata.get("Header 3"):
        md_split_docs.page_content = f'### {md_split_docs.metadata.get("Header 3", "")}\n\n{md_split_docs.page_content}'
    if md_split_docs.metadata.get("Header 2"):
        md_split_docs.page_content = f'## {md_split_docs.metadata.get("Header 2", "")}\n\n{md_split_docs.page_content}'
    if md_split_docs.metadata.get("Header 1"):
        md_split_docs.page_content = f'# {md_split_docs.metadata.get("Header 1", "")}\n\n{md_split_docs.page_content}'

def load_man_united_data():
    united_data_dir = Path("man_united_data").resolve()
    email_files: list[Document] = []
    for directory in DIRECTORIES.keys():
        for markdown in (united_data_dir / directory).glob("*.md"):
            with open(markdown, 'r') as file:
                content = file.read()
                match = re.search(
                    r"^[ \t-]*source:[ \t]*\[[^\]\n]*\]\((https?://[^)\s]+)\)",
                    content,
                    flags=re.IGNORECASE | re.MULTILINE,
                )
                md_docs = markdown_splitter.split_text(content)
                for md_split_docs in md_docs:
                    md_split_docs.metadata = md_split_docs.metadata | {
                        "path": f'{directory}/{markdown.name}',
                        "source": match.group(1) if match else '',
                        "type": DIRECTORIES[directory].value,
                        "len": len(content)
                    }

                email_files.extend(md_docs)

    return email_files
email_files = load_man_united_data()
email_files

[Document(metadata={'Header 1': 'Tickets and tours', 'path': 'homepage/tickets-and-tours.md', 'source': 'https://www.manutd.com/en', 'type': 'homepage', 'len': 1271}, page_content="Source: [https://www.manutd.com/en](https://www.manutd.com/en)  \nThe homepage displayed these offers and entry points on 4 October 2026. Follow each link for current prices and availability.  \n- [Football at the Theatre of Dreams Be here in 2026/27](https://tickets.manutd.com/en-GB/categories/home-tickets?web)\n- [Red Planet Immersive Experience at Old Trafford. Museum & Tour Red Planet Immersive Experience at Old Trafford.](https://tickets.manutd.com/custom/calendar.aspx?showid=c9005703-f55b-6e8f-700e-019ff661dcea)\n- [Support the Reds in Europe Champions League tickets](https://tickets.manutd.com/en-GB/categories/home-tickets?t=y&utm_source=manutd_app&utm_medium=ad_card&utm_campaign=match_tickets&utm_content=cl202627)\n- [Support Eva Olid's Reds! United Women v Liverpool on Saturday](https://tickets.manu

In [439]:
text_split_email_files = text_splitter.split_documents(email_files)
len(text_split_email_files)

2834

In [440]:
for doc in text_split_email_files:
    load_pagecontent_with_headers(doc)

In [380]:
store.add_documents(text_split_email_files)

['9cb816b0-75d1-4215-a4df-94c6d6596575',
 '609dfa4e-f7ea-482d-9ed2-2b660fdeb700',
 '5f36c278-5b74-4453-aa4c-cc64db20e6bd',
 '18096541-6622-4dd4-b820-3e6b633e5d21',
 'bc152f53-26c7-490f-8d4b-e85e9698e66f',
 'd6083c73-b570-4c65-ade2-6a9cd15341fa',
 'f2dd9b0f-8cc4-47ea-99b4-53247b67de41',
 '37882f71-5abc-490a-a631-cd87a24ebf61',
 '61e2d5a7-cc05-45ff-981e-25ae7fc9dc8a',
 'c011bed3-089c-4518-bb04-957d072f8169',
 '181b837c-4c7c-4759-9315-bcb9c990fced',
 '143a1635-1243-4951-9b2c-0f86e02837eb',
 '31ab23a8-47fc-43d6-80d8-7c13dead7a2d',
 'd74eda3e-03be-4065-b3d8-5a079faebb82',
 '7bd9fb81-1701-47c0-819c-ff4756bf381f',
 '1bd12d50-94ca-4834-b32f-466588d52720',
 'd50a3b88-704e-4a04-9bd1-6cb53b21399f',
 '921fccca-0eb8-46e2-8ad6-ccf2da07641b',
 'f646b82d-f18d-4b77-b9cb-4d079295f7e5',
 'f296d3d5-a1ff-46c7-a07f-b3ff830075b1',
 'ef4eb2e8-acbb-4bc6-b9f0-4d960bb65132',
 'e43b7767-2e92-458a-b0ba-7feeede3c608',
 '9b647005-f1b3-4b10-a620-4b02efd8c5c0',
 'ad6c657a-a7f2-48c6-9f69-5e3a2376bba4',
 '48e993b3-0166-

In [441]:
DB_STORE_URI = (
    f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}"
    f":{POSTGRES_PORT}/{POSTGRES_DB}"
)
stack = ExitStack()
postgres_saver = stack.enter_context(PostgresSaver.from_conn_string(DB_STORE_URI))
postgres_saver.setup()
postgres = stack.enter_context(PostgresStore.from_conn_string(DB_STORE_URI))
postgres.setup()
backend = StoreBackend(
    store=postgres,
    namespace=lambda rt: ("man_utd_backend", rt.execution_info.thread_id, ),
)

In [442]:
def define_backend_file_path(thread_id: str, file_path: str) -> Path:
    return Path(f'/retrieved/{thread_id}/{file_path}')

@tool(parse_docstring=True)
def search_manchester_united_documentation(
        query: str,
        source: Optional[SourcePydantic],
        runtime: ToolRuntime[Context]
):
    """
    Search documentation about Manchester United in the vector store using similarity search with the query.

    This method uses our source of knowledge where we stored all the information related to the Manchester United
    webpage. Use this method when you have to get all relative information using the user query or even an enhanced
    query you create in order to be more accurate in the documents retrieved.

    This method is going to use similarity search, and it is going to get 8 documents and store it into the backend
    that you will have to read on your own to get the information. As a result, this method is going to return a
    string with the file path that is going to be stored every file and the metada of that file:
    - source: link of the team's web, if it is provided
    - path: relative file path where the entire Markdown file stored the info chunk

    Args:
        query: Natural language search query. Could be the user query directly or an enhanced query to improve the
        search quality
        source: Optional source param in order to apply filters and perform similarity search only in those specific
        chunks of data whose metadata matches the type
        runtime: Agent implicit runtime

    Returns:
        File paths where retrieved chunks were saved under /retrieved/, as well as the source link and the Markdown
        file path that contain the whole info
    """
    k = 8
    optional_filter = { "type": source.type.value } if source else None
    documents = store.similarity_search_with_score(query=query, k=k, filter=optional_filter)
    result = f"Here are {k} documents with its path in the backend and source link, if it is present.\n\n"
    files: list[tuple[str, bytes]] = []
    thread_id = runtime.execution_info.thread_id if runtime.execution_info.thread_id else ""
    for (document, score) in documents:

        path = define_backend_file_path(thread_id, document.metadata.get("path", ""))
        path = path.with_name(f'{document.id}-{path.name}')

        source = ""
        if document.metadata.get("source"):
            source = f'Source link: {document.metadata.get("source", "")}'
        files.append((str(path), document.page_content.encode('utf-8')))
        result += f"""
        - Path: {path}
        {source}

        --------------------------------------

        """

    backend.upload_files(files)
    return result


class ThreadId(BaseModel):
    thread_id: str
class RuntimeContext(BaseModel):
    execution_info: ThreadId

runtime = RuntimeContext(execution_info=ThreadId(thread_id="user id"))


In [443]:
@tool(parse_docstring=True)
def search_manchester_united_files(file_paths: list[str], runtime: ToolRuntime[Context]):
    """
    This method extracts the entire document of the file path array given by the parameter and stores its content
    into the backend available for the agent.

    This method is useful when agent knows exactly which are the files needed to accomplish the task, because the
    user query is really straightforward or the agent considers that it needs exactly some set of files.

    Args:
        file_paths: File path array to extract data and store it into the backend to make it available to the agent
        runtime: Agent implicit runtime

    Returns:
        File paths where retrieved document were saved under /retrieved/. If any file exists or any path is returned,
        then those specific or any file exists.
    """
    thread_id = runtime.execution_info.thread_id if runtime.execution_info.thread_id else ""

    store_documents = store.get(where={"path": {"$in": file_paths}})
    result = f"Here are the documents that match the file paths given by parameter.\n\n"
    files: list[tuple[str, bytes]] = []
    for file_path in file_paths:
        documents = store_documents.get("documents", [])
        metadatas = store_documents.get("metadatas", [])
        all_content = "".join([
            content for (content, metadata) in zip(documents, metadatas)
            if metadata["path"] == file_path
        ])
        path = define_backend_file_path(thread_id=thread_id, file_path=file_path)
        files.append((str(path), all_content.encode('utf-8')))
        result += f"- Path of {file_path}: {path}\n"
    backend.upload_files(files)
    return result

In [444]:
# define memory
postgres.put(("users", "memories"), "user_id", {"behaviour": "aggressive"})

In [449]:
def handle_stream(upload_stream: GraphRunStream):
    for name, message in upload_stream.interleave("messages", "tool_calls", "subagents"):
        if name == "tool_calls":
            print()
            print(f"[{datetime.now()}] TOOL CALL: {message.tool_name}({message.input})")
            for delta in message.output_deltas:
                print(delta, end="(end delta)", flush=True)
        elif name == "messages":
            # just for reasoning models
            for msg in message.reasoning:
                print(f"[thinking] {msg}", end="", flush=True)
            for msg in message.text:
                # this stream just prints agent response in streaming
                print(msg, end="", flush=True)
        elif name == "subagents":
            print(f"SUB AGENT {message.name}: ", end="")
            print(f"[{datetime.now()}] SUB AGENT [{message.name}] started")
            for subagent_name, item in message.interleave("messages", "tool_calls"):
                if subagent_name == "messages":
                    print(f"SUB AGENT [{message.name}]", item.text, end="",)
                if subagent_name == "tool_calls":
                    print(f"[{datetime.now()}] SUB AGENT TOOL CALL [{message.name}] {message.tool_name}({message.input})", end="",)
            # print(f"[{message.name}] status: {message.status}. Output: {message.output}")


In [446]:
RESEARCH_SUBAGENT_SYSTEM_PROMPT = """
You are a specialized sub agent, whose task is to read files about Manchester United that are stored in the
source of knowledge. These files may have already been stored in the subagent backend or they need to be searched by
you, depending on the main angent's prompt. After identifying or searched the files, you have to read them and
extract the relevant information based on the user query which is also given by the main agent.

Most of these tasks are going to be reading the files on your own that the main agent has already obtained before
spawning you, letting you the responsibility of gathering the most important information stored in the files that is
related to the user's query.

Other tasks the main agent could deliver to you is to execute `search_manchester_united_files` tool giving you the
file paths and letting you the responsibility of gathering the information and return the most important one.

Also, it is possible that some of the content that you read contain deeper links in the markdown to more expanded information
about the topic. These links are relative to that file and not relative to the root dir, but the main agent is going to give
you the guide to transform the relative file path to the proper file path to use `search_manchester_united_files` tool and
expand the information. However, you needn't expand the information every time you see a link in the files, only with the
files you consider are important relative to the user's query.

You have to provide the main agent only with the important information, not about an opinion related to the topic or
anything that is not about what the user want and the main agent requirements. You can return also the links of the sources
or images that you may encounter during the research task that may be relevant for the user. If you do not find any relevant
info, then you have to communicate it to your main agent.
"""
research_subagent = SubAgent(
    model=model,
    name="Research Subagent",
    description="Use this agent when you need to extract the content of the files stored in the agent backend and return the relevant data according to the user's query. You have to provide this subagent with the file paths that you want to get the significant info from or the specific file path you want this subagent to search using `search_manchester_united_files` and explain.",
    system_prompt=RESEARCH_SUBAGENT_SYSTEM_PROMPT,
    tools=[search_manchester_united_files],
    skills=[]
)


In [448]:
SYSTEM_PROMPT = """
You are and specialized agent in Manchester united webpage. Your only task is to give to the user an according
response of his question using proper information about Manchester united using the source of knowledge that we have
in our database as the context. Then, your goal is to use this info to respond accurately to the user in a way that
does not seem that you've just read some files and copy paste the information to the user, but you have digested the
knowledge given and articulated the information in an elaborated response that combines and relates the information.

As our source of knowledge, we have several different type, based on the topic scraped in the Manchester united
webpage:

- Homepage: this info is about the page day highlights that are displayed in the main page of the web. Here you can
give a hint to the user that what is the hot topic of the current day in the Manchester United world.
- Matches: here you have everything about Manchester united matches, such as fixtures and results, including the
standings of the team in the current tournament.
- Membership: this topic is about Manchester United membership. There are some question and answers files and a comparison
table.
- News: this topic is about the latest news that happened in Man United world and the details about them, including the
metadata of the piece of news, the overview and the details. This section also has a list of news grouped by the news page.
- People: this info is about people that are involved in Manchester United F.C., such as women players, men players and staff.
You also have the list as well as the detail related to everyone.
- Store: Manchester united ecommerce and store the showcase in their webpage. This topic is related to any kind of product
that is showed in the webpage for only men and women.
- Travel: This topic is about getting to the stadium and how is the commute to the field.
- Videos: This section has all the videos that have been published in the Manchester United webpage.

All these different topics have their own skill that you have to read every time YOU consider the user want to get to
know about any of this topic in order to know which are the next steps you have to get to obtain the response. If you
are not sure which is the topic the user wants to know about, lean into the news topic because maybe it is related to
some piece of news.

These skills are going to explain you how to get the files or how to get the source of knowledge available to the sub agent to
read. There are mostly 2 ways of fetching the source of knowledge that are described in the skills: call
`search_manchester_united_documentation` and spawn the sub agent with returned file paths or spawning directly the sub agent
after realizing with the skill that the user's query may be solved with a specific file (or both). Eventually you are going
to spawn the subagent in order to read the files and not overload the context.

Perhaps the user wants to know different topics at the same time, then you will have to read different skills, start
the subagent making sure its going to read the necessary files of the different topics and return you the most important data
extracted. Then, you must gather all the info and build a dynamic and elaborated response depending on the context given by
the user.

If you do not get any reliable context from our database, then, since you are an AI system powered by an LLM, you
may have the response in your own knowledge, such as historical facts. Otherwise you do not get any data from the
subagent and you consider the user wants to know something unrelated to the team, then you must explain them that you
are a Man United bot with no other purpose (in different ways every time the user asks for something else). YOU
CANNOT GIVE ANY ANSWER THAT IS NOT RELATED TO MANCHESTER UNITED.

The response you have to give is dynamic and combining all the data you have to deliver an organic response that feels
natural. If the subagent gave you some image link you can also insert it into the response, as well as the links to the videos
or the sources.
"""
DB_STORE_URI = (
    f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}"
    f":{POSTGRES_PORT}/{POSTGRES_DB}"
)

agent = create_deep_agent(
    name="Manchester United Orchestrator",
    model=model,
    permissions=[
        FilesystemPermission(operations=["write"], paths=["/skills/**"], mode="deny")
    ],
    skills=["/skills/"],
    backend=CompositeBackend(
        default=backend,
        routes= {
            "/skills/": FilesystemBackend(Path("skills").resolve(), virtual_mode=True)
        }
    ),
    context_schema=Context,
    tools=[search_manchester_united_documentation],
    store=postgres,
    subagents=[research_subagent],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=postgres_saver
)

agent_stream = agent.stream_events(
    input={
        "messages": [
            HumanMessage(
                content="Well tell me a brief history about Antony in the Manchester United"
            )
        ]
    },
    config=RunnableConfig(
        recursion_limit=100,
        configurable={"thread_id": "4"}
    ),
    context=Context(user_id="user_id"),
    version="v3"
)
handle_stream(agent_stream)


[2026-10-20 12:07:40.587560] TOOL CALL: ls({'path': '/skills/people'})

[2026-10-20 12:07:43.293014] TOOL CALL: read_file({'file_path': '/skills/people/SKILL.md', 'offset': 0, 'limit': 1000})

[2026-10-20 12:07:45.802534] TOOL CALL: search_manchester_united_documentation({'query': 'Antony Manchester United career joined Ajax September 2022 debut Arsenal goals Real Betis transfer', 'source': None})

[2026-10-20 12:07:52.350056] TOOL CALL: task({'description': "Research only: User asks brief history of Antony at Manchester United. Semantic search returned irrelevant players. Locate Antony-specific club profile or news records in available backend using filesystem tools (ls/glob/grep; search_manchester_united_files if available). Verify joining date, debut, key moments, trophies and departure if documented. Return concise facts and sources. Do not confuse Antony with Anthony or other players. If no Antony records found state so; don't infer facts from unrelated returned profiles.", 'suba

In [ ]:
stack.close()